In [ ]:
import os
import numpy as np
from numpy.random import randn, randint
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten, Conv2D, Conv2DTranspose, LeakyReLU, Dropout
from matplotlib import pyplot

# Ayrımcı
def ayrimci(in_shape=(32,32,3)):
    model = Sequential()
    model.add(Conv2D(64, (3,3), padding='same', input_shape=in_shape))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2D(256, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Flatten())
    model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid'))
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    return model

# Üretici
def uretici(son_boyut):
    model = Sequential()
    n_nodes = 256 * 4 * 4
    model.add(Dense(n_nodes, input_dim=son_boyut))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Reshape((4, 4, 256)))
    model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2D(3, (3,3), activation='tanh', padding='same'))
    return model

# GAN
def GANs(g_model, d_model):
    d_model.trainable = False
    model = Sequential([g_model, d_model])
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt)
    return model

# Veri yükleme
def Veri_set_yukleme():
    (trainX, _), (_, _) = cifar10.load_data()
    X = trainX.astype('float32')
    X = (X - 127.5) / 127.5
    return X

# Gerçek örnekler
def ornek_veri(dataset, n_samples):
    ix = randint(0, dataset.shape[0], n_samples)
    X = dataset[ix]
    y = np.ones((n_samples, 1))
    return X, y

# Gürültü
def noktalar_olusturma(son_boyut, n_samples):
    x_input = randn(son_boyut * n_samples)
    return x_input.reshape(n_samples, son_boyut)

# Sahte veri
def sahte_nesne_olusturma(g_model, son_boyut, n_samples):
    x_input = noktalar_olusturma(son_boyut, n_samples)
    X = g_model.predict(x_input, verbose=0)
    y = np.zeros((n_samples, 1))
    return X, y

# Görselleştirme
def cizim(examples, epoch, n=7):
    examples = (examples + 1) / 2.0
    for i in range(n * n):
        pyplot.subplot(n, n, 1 + i)
        pyplot.axis('off')
        pyplot.imshow(examples[i])
    filename = f"/content/gan_ornek_{epoch+1:03d}.png"
    pyplot.savefig(filename)
    pyplot.close()

# Değerlendirme
def degerlendir(epoch, g_model, d_model, dataset, son_boyut, n_samples=150):
    X_gercek, y_gercek = ornek_veri(dataset, n_samples)
    _, acc_gercek = d_model.evaluate(X_gercek, y_gercek, verbose=0)
    X_sahte, y_sahte = sahte_nesne_olusturma(g_model, son_boyut, n_samples)
    _, acc_sahte = d_model.evaluate(X_sahte, y_sahte, verbose=0)
    print(f"Gerçek doğruluğu: {acc_gercek*100:.2f}% | Sahte doğruluğu: {acc_sahte*100:.2f}%")
    cizim(X_sahte, epoch)
    g_model.save(f"/content/gan_model_{epoch+1:03d}.h5")

# Eğitim
def train(g_model, d_model, gan_model, dataset, son_boyut, n_epochs=30, n_batch=128):
    bat_per_epo = int(dataset.shape[0] / n_batch)
    half_batch = int(n_batch / 2)

    for i in range(n_epochs):
        for j in range(bat_per_epo):
            X_gercek, y_gercek = ornek_veri(dataset, half_batch)
            d_loss1, _ = d_model.train_on_batch(X_gercek, y_gercek)

            X_sahte, y_sahte = sahte_nesne_olusturma(g_model, son_boyut, half_batch)
            d_loss2, _ = d_model.train_on_batch(X_sahte, y_sahte)

            X_gan = noktalar_olusturma(son_boyut, n_batch)
            y_gan = np.ones((n_batch, 1))
            g_loss = gan_model.train_on_batch(X_gan, y_gan)

            print(f">{i+1}, {j+1}/{bat_per_epo}, dr={d_loss1:.3f}, ds={d_loss2:.3f}, g={g_loss:.3f}")

        if (i+1) % 5 == 0:
            degerlendir(i, g_model, d_model, dataset, son_boyut)

# Çalıştır
son_boyut = 100
d_model = ayrimci()
g_model = uretici(son_boyut)
gan_model = GANs(g_model, d_model)
dataset = Veri_set_yukleme()

train(g_model, d_model, gan_model, dataset, son_boyut)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:83: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


>1, 1/390, dr=0.682, ds=0.688, g=0.693
>1, 2/390, dr=0.686, ds=0.688, g=0.692
>1, 3/390, dr=0.688, ds=0.689, g=0.692
>1, 4/390, dr=0.688, ds=0.689, g=0.691
>1, 5/390, dr=0.688, ds=0.689, g=0.690
>1, 6/390, dr=0.689, ds=0.690, g=0.688
>1, 7/390, dr=0.689, ds=0.691, g=0.686
>1, 8/390, dr=0.690, ds=0.692, g=0.684
>1, 9/390, dr=0.691, ds=0.694, g=0.680
>1, 10/390, dr=0.693, ds=0.696, g=0.676
>1, 11/390, dr=0.695, ds=0.699, g=0.670
>1, 12/390, dr=0.699, ds=0.703, g=0.664
>1, 13/390, dr=0.702, ds=0.707, g=0.657
>1, 14/390, dr=0.706, ds=0.711, g=0.650
>1, 15/390, dr=0.710, ds=0.715, g=0.643
>1, 16/390, dr=0.714, ds=0.720, g=0.636
>1, 17/390, dr=0.719, ds=0.724, g=0.629
>1, 18/390, dr=0.723, ds=0.728, g=0.622
>1, 19/390, dr=0.727, ds=0.733, g=0.616
>1, 20/390, dr=0.732, ds=0.737, g=0.609
>1, 21/390, dr=0.736, ds=0.741, g=0.603
>1, 22/390, dr=0.740, ds=0.746, g=0.597
>1, 23/390, dr=0.744, ds=0.749, g=0.592
>1, 24/390, dr=0.748, ds=0.753, g=0.586
>1, 25/390, dr=0.752, ds=0.757, g=0.581
>1, 26/39

>6, 1/390, dr=1.342, ds=1.409, g=0.245
>6, 2/390, dr=1.272, ds=1.330, g=0.245
>6, 3/390, dr=1.242, ds=1.288, g=0.245
>6, 4/390, dr=1.223, ds=1.260, g=0.245
>6, 5/390, dr=1.209, ds=1.243, g=0.245
>6, 6/390, dr=1.201, ds=1.230, g=0.245
>6, 7/390, dr=1.194, ds=1.220, g=0.245
>6, 8/390, dr=1.189, ds=1.213, g=0.245
>6, 9/390, dr=1.185, ds=1.207, g=0.245
>6, 10/390, dr=1.182, ds=1.202, g=0.245
>6, 11/390, dr=1.180, ds=1.198, g=0.245
>6, 12/390, dr=1.177, ds=1.194, g=0.245
>6, 13/390, dr=1.175, ds=1.191, g=0.245
>6, 14/390, dr=1.174, ds=1.189, g=0.245
>6, 15/390, dr=1.173, ds=1.187, g=0.245
>6, 16/390, dr=1.172, ds=1.185, g=0.245
>6, 17/390, dr=1.170, ds=1.183, g=0.245
>6, 18/390, dr=1.169, ds=1.181, g=0.245
>6, 19/390, dr=1.169, ds=1.180, g=0.245
>6, 20/390, dr=1.168, ds=1.179, g=0.245
>6, 21/390, dr=1.167, ds=1.177, g=0.245
>6, 22/390, dr=1.166, ds=1.176, g=0.245
>6, 23/390, dr=1.166, ds=1.175, g=0.245
>6, 24/390, dr=1.166, ds=1.175, g=0.245
>6, 25/390, dr=1.165, ds=1.174, g=0.245
>6, 26/39